<a href="https://colab.research.google.com/github/ibrahimshkhess/smart-air-quality-lighting-iot/blob/main/AirModelPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AirModelPrediction

In [32]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).



# Library setting



In [33]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LSTM
from sklearn.preprocessing import StandardScaler
import os
import random
import keras


Seeting the seeds:

In [34]:
# fix random seed for reproducibility
def set_all_seeds(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

set_all_seeds(42)
np.random.seed(42) #it also fix the seed for sklearn

# Inizialization of Input Data

In [55]:
driver_path='/content/drive/MyDrive/AirQualityPrediction/'

In [56]:
air_quality_raw = pd.read_csv(driver_path+'air_mesurament_adjusted.csv', header=None, index_col=False, sep=';', engine='python')

In [57]:
air_quality_df =air_quality_raw.drop(0, axis=0)
cols_data = air_quality_df.columns[0:3]
air_quality_df[cols_data] = air_quality_df[cols_data].apply(pd.to_numeric, errors='coerce')
col_timestamp  = air_quality_df.columns[3]
air_quality_df[col_timestamp] = pd.to_datetime(
    air_quality_df[col_timestamp],
    format='%d/%m/%Y %H:%M',
    errors='coerce'
)
air_quality_df = air_quality_df.dropna(axis=0)
print(air_quality_df.head(), air_quality_df.shape)

         0        1        2                   3
1  397.163  419.985  419.985 2022-10-27 10:55:00
2  396.138  418.902  418.902 2022-10-27 10:55:00
3  397.071  419.888  419.888 2022-10-27 10:55:00
4  397.071  419.888  419.888 2022-10-27 10:55:00
5  397.071  419.888  419.888 2022-10-27 10:55:00 (1048572, 4)


Comprime the dateset taking the average value for each minute

In [63]:
air_quality_df_avg = air_quality_df.groupby(3, as_index=False)[[0, 1, 2]].mean()

,0,1,2,3
0,339.423667,411.451190,411.451048,2022-10-27 10:55:00
1,354.840071,374.477714,360.905714,2022-10-27 10:56:00
2,324.723963,360.563556,360.535333,2022-10-27 10:57:00
3,360.467074,342.245259,355.533630,2022-10-27 10:58:00
4,358.728444,327.283000,341.545519,2022-10-27 10:59:00


In [65]:
air_quality_df_avg.tail(10)

,0,1,2,3
38538,253.223000,254.310407,227.643815,2022-11-24 00:11:00
38539,301.256429,308.571929,298.939250,2022-11-24 00:12:00
38540,240.335074,311.740185,311.740037,2022-11-24 00:13:00
38541,267.772222,283.362000,294.201037,2022-11-24 00:14:00
38542,294.681963,300.751889,290.236296,2022-11-24 00:15:00
38543,244.142179,267.435536,267.435321,2022-11-24 00:16:00
38544,292.575852,264.819519,287.317370,2022-11-24 00:17:00
38545,265.977519,300.492778,300.492741,2022-11-24 00:18:00
38546,265.934148,270.944963,261.589444,2022-11-24 00:19:00
38547,250.808444,284.318741,284.318667,2022-11-24 00:20:00
